# nflbowl — Colab GPU quickstart

Trains the primary 12F encoder (F4, physics-informed contrastive + matched-state alignment) on a Colab GPU, evaluates it and saves every artifact to Google Drive.

**Before you start**
1. *Runtime → Change runtime type → GPU* (T4 is enough).
2. Put the data bundle on Drive, e.g. `MyDrive/NFLBowl/bundles/m12f_v1/` (built on the research machine by `01_data_preparation.ipynb`).
3. Replace `YOUR_USERNAME/YOUR_REPO` in Cell 1 (a private repo needs a token: see `docs/nflbowl/COLAB.md`).

**After a disconnect / runtime reset:** run Cells 1–4 again with the SAME `name=`; `exp.train(..., resume=True)` continues from the last epoch checkpoint on Drive, and finished phases are reused (see the last cell).

In [ ]:
# Cell 1 — install (keeps Colab's own PyTorch/CUDA build; nothing torch-related is installed)
%pip install -q "git+https://github.com/YOUR_USERNAME/YOUR_REPO.git"

In [ ]:
# Cell 2 — GPU detection
from nflbowl import Experiment

info = Experiment.device_info()

In [ ]:
# Cell 3 — persistent storage
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# Cell 4 — initialise (re-running this with the same name REOPENS the experiment)
exp = Experiment(
    data_dir="/content/drive/MyDrive/NFLBowl/bundles/m12f_v1",   # staged to /content/nflbowl_stage by the colab_gpu preset
    output_dir="/content/drive/MyDrive/NFLBowl",                 # artifacts persist on Drive
    config="colab_gpu",
    device="auto",
    name="f4_main",
)
exp

In [ ]:
# Cell 5 — prepare data (stage to local disk, validate, cache derived targets + F0 features)
exp.prepare()

In [ ]:
# Cell 6 — train the primary encoder (resumes from Drive checkpoints if interrupted)
exp.train("F4", resume=True)

In [ ]:
# Cell 7 — embeddings + evaluation (standard = the 12F evaluation row; baselines for context)
exp.embed("F4")
exp.evaluate("F0", level="standard")
exp.evaluate("random", level="standard")
exp.evaluate("F4", level="standard")
exp.compare(level="standard")

In [ ]:
# Cell 8 — figures, report, export (all written under the experiment directory on Drive)
figs = exp.plot()
print(exp.report())
print(exp.export())

## Reconnecting in a new session

```python
%pip install -q "git+https://github.com/YOUR_USERNAME/YOUR_REPO.git"
from google.colab import drive; drive.mount("/content/drive")
from nflbowl import Experiment
exp = Experiment.open("/content/drive/MyDrive/NFLBowl/experiments/f4_main",
                      runtime={"stage_dir": "/content/nflbowl_stage"})
exp.prepare()                 # re-stages the bundle; derived tables come from the Drive cache
exp.resume()                  # continues any training job left 'running'
exp.embed("F4", reuse=True)   # loads saved embeddings — no recomputation
exp.evaluate("F4", reuse=True)
```